# Collect a small multi-modal dataset

Run a few random steps on several environments using **mouse-env** and save them into a `DatasetStore`.

| Environment       | Observation | Action    |
|-------------------|-------------|-----------|
| FrozenLake, Taxi  | discrete    | discrete  |
| CartPole          | continuous  | discrete  |
| Pendulum          | continuous  | continuous|
| Breakout (Atari)  | image 84×84 | discrete  |


In [1]:
import ale_py
import gymnasium as gym
from gymnasium.wrappers import AtariPreprocessing

from mouse_envs import EnvConfig, make_vector_env
from mouse_core.data import DatasetStore, push_stores_to_hub

gym.register_envs(ale_py)  # register the ALE/* Atari environments for env_fn factories


## The environments

We describe each environment with an `EnvConfig`.
mouse-env builds it, handles resets, and gives back clean step results.


In [2]:
IMG_SIZE = 84

# We collect the exact same tiny number of steps from every environment.
STEPS_PER_ENV = 5

def make_atari():
    """Build a fresh Atari env with standard preprocessing."""
    env = gym.make("ALE/Breakout-v5", frameskip=1, max_episode_steps=10000)
    return AtariPreprocessing(env, grayscale_obs=True, scale_obs=False,
                              screen_size=IMG_SIZE, frame_skip=4, noop_max=30)

# One config per environment. mouse-env will step them for us.
ENV_CONFIGS = [
    EnvConfig("FrozenLake-v1", seed=0, num_envs=1, max_episode_steps=100,
              kwargs={"is_slippery": True}),
    EnvConfig("Taxi-v4",        seed=0, num_envs=1, max_episode_steps=100),
    EnvConfig("CartPole-v1",    seed=0, num_envs=1, max_episode_steps=100),
    EnvConfig("Pendulum-v1",    seed=0, num_envs=1, max_episode_steps=100),
    EnvConfig("ALE/Breakout-v5", seed=0, num_envs=1, max_episode_steps=10000,
              observation_kind="image", env_fn=make_atari),
]

## Collect a few steps

Call `step(actions)`. Get back results.
Attach the action we just used, put it first, and append the row.

In [3]:
def result_to_row(result, action):
    return {"action": action["action"], **result}


def collect_steps(store, cfg, num_steps):
    """Run a few steps and save (action, result) rows."""
    env = make_vector_env(cfg)
    for _ in range(num_steps):
        actions = env.sample_random_actions()
        results, _ = env.step(actions)
        row = result_to_row(results[0], actions[0])
        row["env"] = cfg.group_id
        store.append(row)
    env.close()
    return num_steps

# Collect steps. We also tag each row with its env so the table is easy to read.
store = DatasetStore()

for cfg in ENV_CONFIGS:
    n_steps = collect_steps(store, cfg, STEPS_PER_ENV)
    print(f"{cfg.group_id}: {n_steps} steps (store now has {len(store)} total)")

print(store)

FrozenLake-v1: 5 steps (store now has 5 total)
Taxi-v4: 5 steps (store now has 10 total)
CartPole-v1: 5 steps (store now has 15 total)
Pendulum-v1: 5 steps (store now has 20 total)


A.L.E: Arcade Learning Environment (version 0.12.0+0706845)
[Powered by Stella]


ALE/Breakout-v5: 5 steps (store now has 25 total)
DatasetStore(steps=25)


## Push to the Hugging Face Hub

To upload, set `REPO_ID` to something like `"your-org/my-mouse-dataset"`.


In [4]:
REPO_ID = "mouse-core-test"  # e.g. "your-org/my-mouse-dataset"

if REPO_ID:
    # config_name creates a "bin" inside the dataset repo.
    # push deletes the old README so a fresh one is written.
    url = push_stores_to_hub(
        [store], repo_id=REPO_ID, split="train", config_name="demo_multi_modal"
    )
    print(f"Pushed to {url}")
else:
    print("Set REPO_ID to upload this dataset to the Hugging Face Hub.")


Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Pushed to micahr234/mouse-core-test (train: 25 steps)
Pushed to https://huggingface.co/datasets/micahr234/mouse-core-test
